In [2]:
import pandas as pd
import numpy as np
import folium
import geopandas
import os

### Óbitos em Sao Paulo

In [7]:
obitos_sp = pd.read_csv('../dados/SP/obitos_publico.csv', delimiter=';', encoding='unicode_escape')

In [8]:
obitos_sp.head()

,Id da Delegacia (RDO),Número do Bo (RDO),Ano do BO (RDO),Data do Óbito,Data do Acidente,Tipo de via,Dia do óbito,Mês do Óbito,Mês Óbito Descrição,Ano do Óbito,...,Tipo do veículo da vítima,Tipo de vítima,Local do óbito,Tipo de acidente,Sub Tipo do Acidente,Sexo,Faixa etária,Idade da vítima,Outro Veículo Envolvido,Tempo entre o Acidente e o Óbito
0,120300,1005409502,2023,2023-04-30,2023-04-30,Vias Municipais,30,4,ABRIL,2023,...,MOTOCICLETA,CONDUTOR,VIA,CHOQUE,ÁRVORE,MASCULINO,18-24,22,NÃO HÁ,Até 30 dias
1,100907,1005405590,2023,2023-04-30,2022-11-24,Vias Municipais,30,4,ABRIL,2023,...,MOTOCICLETA,CONDUTOR,ESTABELECIMENTO DE SAUDE,COLISAO,FRONTAL,MASCULINO,18-24,18,AUTOMOVEL,Mais de 30 dias
2,120505,1005409294,2023,2023-04-30,2023-04-30,Vias Municipais,30,4,ABRIL,2023,...,MOTOCICLETA,CONDUTOR,ESTABELECIMENTO DE SAUDE,ATROPELAMENTO,ANIMAL,MASCULINO,40-44,42,NÃO HÁ,Até 30 dias
3,10326,1005403023,2023,2023-04-30,2023-04-30,Vias Municipais,30,4,ABRIL,2023,...,MOTOCICLETA,CONDUTOR,VIA,COLISAO,LATERAL,MASCULINO,35-39,35,AUTOMOVEL,Até 30 dias
4,10211,1005402331,2023,2023-04-30,2023-04-30,Vias Municipais,30,4,ABRIL,2023,...,MOTOCICLETA,CONDUTOR,ESTABELECIMENTO DE SAUDE,OUTROS,TOMBAMENTO VEÍCULO DE 2 RODAS,MASCULINO,18-24,20,NÃO HÁ,Até 30 dias


In [39]:
obitos_sp.columns

Index(['Id da Delegacia (RDO)', 'Número do Bo (RDO)', 'Ano do BO (RDO)',
       'Data do Óbito', 'Data do Acidente', 'Tipo de via', 'Dia do óbito',
       'Mês do Óbito', 'Mês Óbito Descrição', 'Ano do Óbito',
       'Ano/Mês do Óbito', 'Dia da Semana', 'Hora do Acidente', 'Turno',
       'Município', 'Região Administrativa', 'Logradouro', 'Númeral / KM',
       'Jurisdição', 'Administração', 'Conservação',
       'Tipo do local da ocorrência', 'LAT_(GEO)', 'LONG_(GEO)',
       'Tipo do veículo da vítima', 'Tipo de vítima', 'Local do óbito',
       'Tipo de acidente', 'Sub Tipo do Acidente', 'Sexo', 'Faixa etária',
       'Idade da vítima', 'Outro Veículo Envolvido',
       'Tempo entre o Acidente e o Óbito'],
      dtype='object')

In [50]:
obitos_sp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46174 entries, 0 to 46173
Data columns (total 34 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Id da Delegacia (RDO)             46174 non-null  int64  
 1   Número do Bo (RDO)                46174 non-null  int64  
 2   Ano do BO (RDO)                   46174 non-null  int64  
 3   Data do Óbito                     46174 non-null  object 
 4   Data do Acidente                  46174 non-null  object 
 5   Tipo de via                       46174 non-null  object 
 6   Dia do óbito                      46174 non-null  int64  
 7   Mês do Óbito                      46174 non-null  int64  
 8   Mês Óbito Descrição               46174 non-null  object 
 9   Ano do Óbito                      46174 non-null  int64  
 10  Ano/Mês do Óbito                  46174 non-null  float64
 11  Dia da Semana                     46174 non-null  object 
 12  Hora

In [51]:
obitos_sp['Ano do BO (RDO)'].unique()

array([2023, 2022, 2021, 2020, 2019, 2012, 2015, 2016, 2018, 2017, 2014,
       2013, 2011], dtype=int64)

### Acidentes Nao Fatais

In [10]:
nao_fatais = pd.read_csv("../dados/SP/acidentes_naofatais.csv", delimiter=";", encoding='unicode_escape')

In [55]:
nao_fatais.head()

,ID,Data do Acidente,Dia do Acidente,Mês do Acidente,Ano do Acidente,Ano/Mês do Acidente,Dia da Semana,Hora do Acidente,Turno,Município,...,Serviço de Atendimento - PRF,Serviço de Atendimento - Radio Patrulha,Servciço de Atendimento - CPTRAN,Tipo de Via,Tipo de Acidente - Atropelamento (Pedestre),Tipo de Acidente - Atropelamento (Animal),Tipo de Acidente - Choque,Tipo de Acidente - Colisão,Tipo de Acidente - Outros tipos de Acidente,Turno + Dia de Semana
0,9144205,2023-04-30,30,ABRIL,2023,2023.04,DOMINGO,21:45,NOITE,SAO VICENTE,...,NÃO,SIM,NÃO,Vias Municipais,0,0,0,1,0,NOITE DE DOMINGO
1,9144185,2023-04-30,30,ABRIL,2023,2023.04,DOMINGO,20:47,NOITE,ADAMANTINA,...,NÃO,SIM,NÃO,Rodovias,0,0,0,0,1,NOITE DE DOMINGO
2,9144215,2023-04-30,30,ABRIL,2023,2023.04,DOMINGO,21:54,NOITE,GUARULHOS,...,NÃO,SIM,NÃO,Vias Municipais,0,0,0,1,0,NOITE DE DOMINGO
3,9144219,2023-04-30,30,ABRIL,2023,2023.04,DOMINGO,22:11,NOITE,ITAPEVA,...,NÃO,SIM,NÃO,Vias Municipais,0,0,0,1,0,NOITE DE DOMINGO
4,9144220,2023-04-30,30,ABRIL,2023,2023.04,DOMINGO,22:14,NOITE,MONTE APRAZIVEL,...,NÃO,SIM,NÃO,NAO DISPONIVEL,0,0,1,0,0,NOITE DE DOMINGO


In [57]:
nao_fatais.columns

Index(['ID', 'Data do Acidente', 'Dia do Acidente', 'Mês do Acidente',
       'Ano do Acidente', 'Ano/Mês do Acidente', 'Dia da Semana',
       'Hora do Acidente', 'Turno', 'Município', 'Batalhão de Trânsito',
       'Companhia de Trânsito', 'Região Administrativa', 'Logradouro',
       'Numero/KM', 'Jurisdição', 'Administração', 'Conservação', 'LAT_(GEO)',
       'LONG_(GEO)', 'Condições Climáticas', 'Iluminação', 'Mão de direção',
       'Obras na pista', 'Relevo', 'Superfície da via', 'Tipo de pavimento',
       'Tipo de pista', 'Traçado', 'Veículos Envolvidos - Bicicleta',
       'Veículos Envolvidos - Caminhão', 'Veículos Envolvidos - Automóvel',
       'Veículos Envolvidos - Motocicleta', 'Veículos Envolvidos - Ônibus',
       'Veículos Envolvidos - Pedestre', 'Pessoas Envolvidas - Grave',
       'Pessoas Envolvidas - Ileso', 'Pessoas Envolvidas - Leve',
       'Serviço de Atendimento - Bombeiro', 'Serviço de Atendimento - PMRV',
       'Serviço de Atendimento - PRF',
       'Ser

In [58]:
nao_fatais.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 770184 entries, 0 to 770183
Data columns (total 50 columns):
 #   Column                                       Non-Null Count   Dtype  
---  ------                                       --------------   -----  
 0   ID                                           770184 non-null  int64  
 1   Data do Acidente                             770184 non-null  object 
 2   Dia do Acidente                              770184 non-null  int64  
 3   Mês do Acidente                              770184 non-null  object 
 4   Ano do Acidente                              770184 non-null  int64  
 5   Ano/Mês do Acidente                          770184 non-null  float64
 6   Dia da Semana                                770184 non-null  object 
 7   Hora do Acidente                             770184 non-null  object 
 8   Turno                                        770184 non-null  object 
 9   Município                                    770184 non-nul

### Acidentes Fatais

In [11]:
acid_fatais = pd.read_excel("../dados/SP/acidentes_fatais.xlsx")

C:\Users\danil\anaconda3\lib\site-packages\openpyxl\styles\stylesheet.py:226: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [60]:
acid_fatais.head()

,ID,Id Delegacia (RDO),Número BO (RDO),Ano BO (RDO),Data do Acidente,Dia do Acidente,Mês do Acidente,Ano do Acidente,Ano/Mês do Acidente,Dia da semana,...,Iluminação da via (SIOPM),Superfície da Via (SIOPM),Tipo de pista (SIOPM),Outro Veículo Envolvido,Tipo de via,Condições Climáticas (SIOPM),Sentido da Via (SIOPM),Limite da velocidade da via (SIOPM),Quantidade de vítimas,Tempo entre o Acidente e as Mortes
0,3084163,10211,1005402331,2023,2023-04-30,30,ABRIL,2023,2023.04,DOMINGO,...,NAO DISPONIVEL,NAO DISPONIVEL,NAO DISPONIVEL,NÃO HÁ,Vias Municipais,NAO DISPONIVEL,NAO DISPONIVEL,NAO DISPONIVEL,1,Até 30 dias
1,3084303,10216,1005408855,2023,2023-04-30,30,ABRIL,2023,2023.04,DOMINGO,...,NAO DISPONIVEL,NAO DISPONIVEL,NAO DISPONIVEL,AUTOMOVEL,Vias Municipais,NAO DISPONIVEL,NAO DISPONIVEL,NAO DISPONIVEL,1,Até 30 dias
2,3084326,10326,1005403023,2023,2023-04-30,30,ABRIL,2023,2023.04,DOMINGO,...,NAO DISPONIVEL,NAO DISPONIVEL,NAO DISPONIVEL,AUTOMOVEL,Vias Municipais,NAO DISPONIVEL,NAO DISPONIVEL,NAO DISPONIVEL,1,Até 30 dias
3,3084306,30206,1005406617,2023,2023-04-30,30,ABRIL,2023,2023.04,DOMINGO,...,LUZ SOLAR,SECA,DUPLA,NÃO HÁ,Rodovias,BOM,OESTE,Entre 90 e 100km/h,1,Até 30 dias
4,3084305,30406,1005402823,2023,2023-04-30,30,ABRIL,2023,2023.04,DOMINGO,...,NAO DISPONIVEL,NAO DISPONIVEL,NAO DISPONIVEL,AUTOMOVEL,Vias Municipais,NAO DISPONIVEL,NAO DISPONIVEL,NAO DISPONIVEL,1,Até 30 dias


In [61]:
acid_fatais.columns

Index(['ID', 'Id Delegacia (RDO)', 'Número BO (RDO)', 'Ano BO (RDO)',
       'Data do Acidente', 'Dia do Acidente', 'Mês do Acidente',
       'Ano do Acidente', 'Ano/Mês do Acidente', 'Dia da semana',
       'Hora do Acidente', 'Turno', 'Município', 'Região Administrativa',
       'Logradouro', 'Numeral / KM', 'Jurisdição', 'Administração',
       'Conservação', 'Tipo do Local do Acidente', 'Lat (GEO)', 'Long (GEO)',
       'Iluminação da via (SIOPM)', 'Superfície da Via (SIOPM)',
       'Tipo de pista (SIOPM)', 'Outro Veículo Envolvido', 'Tipo de via',
       'Condições Climáticas (SIOPM)', 'Sentido da Via (SIOPM)',
       'Limite da velocidade da via (SIOPM)', 'Quantidade de vítimas',
       'Tempo entre o Acidente e as Mortes'],
      dtype='object')

In [62]:
acid_fatais.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43263 entries, 0 to 43262
Data columns (total 32 columns):
 #   Column                               Non-Null Count  Dtype         
---  ------                               --------------  -----         
 0   ID                                   43263 non-null  int64         
 1   Id Delegacia (RDO)                   43263 non-null  int64         
 2   Número BO (RDO)                      43263 non-null  int64         
 3   Ano BO (RDO)                         43263 non-null  int64         
 4   Data do Acidente                     43263 non-null  datetime64[ns]
 5   Dia do Acidente                      43263 non-null  int64         
 6   Mês do Acidente                      43263 non-null  object        
 7   Ano do Acidente                      43263 non-null  int64         
 8   Ano/Mês do Acidente                  43263 non-null  float64       
 9   Dia da semana                        43263 non-null  object        
 10  Hora do Ac

In [64]:
acid_fatais['Município'].unique()

array(['SAO PAULO', 'ITAPECERICA DA SERRA', 'IBITINGA', 'SARAPUI',
       'IBIUNA', 'SANTOS', 'CUBATAO', 'SANTA MARIA DA SERRA', 'GARCA',
       'ASSIS', 'JAMBEIRO', 'GUARULHOS', 'BERTIOGA', 'SAO SEBASTIAO',
       'AVANHANDAVA', 'AMERICANA', 'JABOTICABAL', 'RIBEIRAO PRETO',
       'CATANDUVA', 'LARANJAL PAULISTA', 'ITAPETININGA', 'SAO VICENTE',
       'PIRACICABA', 'SANTA FE DO SUL', 'SAO JOSE DO RIO PRETO',
       'SOROCABA', 'MATAO', 'CAPAO BONITO', 'COSMOPOLIS',
       'CAMPO LIMPO PAULISTA', 'JUNDIAI', 'PRESIDENTE PRUDENTE',
       'SAO JOSE DOS CAMPOS', 'SAO ROQUE', 'SILVEIRAS', 'CAJATI',
       'PIRAPORA DO BOM JESUS', 'SANTANA DE PARNAIBA', 'OSASCO',
       'PORTO FELIZ', 'SAO BERNARDO DO CAMPO', 'MAUA', 'UBATUBA',
       'CAJAMAR', 'ARARAS', 'OURINHOS', 'ELISIARIO', 'URANIA', 'JALES',
       'JARDINOPOLIS', 'CESARIO LANGE', 'JACAREI', 'VARGEM GRANDE DO SUL',
       'CAMPINAS', 'ALVARES MACHADO', 'OLIMPIA', 'ITUVERAVA',
       'NOVO HORIZONTE', 'INDAIATUBA', 'CACAPAVA', 'SUZANO

In [65]:
sp_acidentes_fatais = acid_fatais[acid_fatais['Município'] == "SAO PAULO"]

In [68]:
loc_acidentes_fatais = sp_acidentes_fatais[['Lat (GEO)','Long (GEO)']]

In [77]:
#Ver como jogar todas as coordenadas no mapa depois
import folium

m = folium.Map(location=[45.372, -121.6972], zoom_start=12, tiles="Stamen Terrain")

tooltip = "Acidente!"

for i,j in loc_acidentes_fatais['Lat (GEO)'],loc_acidentes_fatais['Long (GEO)']:
    folium.Marker(
        [i, j], tooltip=tooltip
    ).add_to(m)

ValueError: too many values to unpack (expected 2)